# Hirano Figure 9: postselection and two-fault audit

This notebook is an executed record of the audit of Hirano *et al.* commit
`c89a53a7a7062b552d31f4c41ae2ba857756359a`. It uses only the vendored authors'
circuit builder, their released Figure 9 counts, the linked spreadsheet export,
and Stim. No circuit or decoder from the surrounding repository is imported.

The `+` experiment is the logical-Z-error probe; the `0` experiment is the
logical-X-error probe.

In [1]:
from pathlib import Path
import inspect
import sys

import numpy as np
import pymatching
import stim
from scipy.optimize import curve_fit

AUDIT_DIR = Path.cwd()
assert (AUDIT_DIR / "fig9_postselection_audit.py").exists(), "Run this notebook from its audit directory"
sys.path.insert(0, str(AUDIT_DIR))

from fig9_postselection_audit import (
    UPSTREAM_COMMIT,
    audit_postselection,
    build_experiment,
    extract_fault_effects,
    print_ambiguity,
    rediscover_ambiguity,
    verify_ambiguity,
)
from false_branch_scaling import P_VALUES, run_point
from true_branch_scaling import power_law as true_power_law, run_job as run_true_job

print(f"upstream commit: {UPSTREAM_COMMIT}")
print(f"Stim: {stim.__version__}")
print(f"PyMatching: {pymatching.__version__}")
print(f"NumPy: {np.__version__}")

upstream commit: c89a53a7a7062b552d31f4c41ae2ba857756359a
Stim: 1.16.0
PyMatching: 2.4.0
NumPy: 2.4.6


## Figure 9 postselection provenance

The next cell reads the chart title from the saved export of the Google Sheet
linked by the authors' plotting script, extracts its released (p=10^{-3})
acceptance rates, builds both branches of the authors' circuit, and samples each
with fixed seeds. The full branch is required to mark every detector for
postselection.

In [2]:
audit_postselection(sample_shots=100_000)

Upstream commit: c89a53a7a7062b552d31f4c41ae2ba857756359a
Spreadsheet chart title: 'full post selection, perfect initialization, surface-distance=5, ZXZ'

+ probe at p=0.001
  released Figure 9 acceptance: 31.789954%


  full_post_selection=False: detectors=231, postselected=40
    sampled acceptance (100,000 shots): 60.701000%


  full_post_selection=True: detectors=135, postselected=135
    sampled acceptance (100,000 shots): 31.753000%

0 probe at p=0.001
  released Figure 9 acceptance: 31.885245%


  full_post_selection=False: detectors=231, postselected=40
    sampled acceptance (100,000 shots): 60.777000%


  full_post_selection=True: detectors=135, postselected=135
    sampled acceptance (100,000 shots): 31.737000%


## Scaling of the authors' non-full branch

This cell directly constructs and samples the authors' exact
`full_post_selection=False` circuit with its postselection mask and PyMatching
decoder. Sampling continues to 500 logical errors or five million shots at
each of the eight Figure 9 physical-error values. The quoted exponent is an
effective finite-window log-log fit; it is distinct from the asymptotic order
established by fault enumeration below.

In [3]:
scaling_results = {}
for probe_index, (probe, error_name) in enumerate((("+", "Z"), ("0", "X"))):
    rows = []
    print(f"{error_name}-error probe ({probe}):")
    for p_index, p in enumerate(P_VALUES):
        shots, accepted, errors = run_point(
            probe,
            float(p),
            target_errors=1000,
            max_shots=10_000_000,
            batch_shots=100_000,
            seed=10_000 + probe_index * 100 + p_index,
        )
        rate = errors / accepted
        rows.append((float(p), shots, accepted, errors, rate))
        print(
            f"  p={p:.4g}, shots={shots:,}, accepted={accepted:,}, "
            f"errors={errors}, acceptance={accepted / shots:.6%}, pL={rate:.10g}",
            flush=True,
        )
    scaling_results[probe] = rows
    p_values = np.array([row[0] for row in rows])
    rates = np.array([row[4] for row in rows])
    error_counts = np.array([row[3] for row in rows])
    unweighted = np.polyfit(np.log(p_values), np.log(rates), 1)
    weighted = np.polyfit(
        np.log(p_values),
        np.log(rates),
        1,
        w=np.sqrt(error_counts),
    )
    print(f"{error_name}-error fits:")
    print(f"  unweighted: pL ≈ {np.exp(unweighted[1]):.3g} p^{unweighted[0]:.3f}")
    print(f"  error-weighted: pL ≈ {np.exp(weighted[1]):.3g} p^{weighted[0]:.3f}")


Z-error probe (+):


  p=0.0005, shots=10,000,000, accepted=7,782,274, errors=487, acceptance=77.822740%, pL=6.25781102e-05


  p=0.0006, shots=10,000,000, accepted=7,404,910, errors=675, acceptance=74.049100%, pL=9.115573316e-05


  p=0.0008, shots=7,500,000, accepted=5,023,942, errors=1013, acceptance=66.985893%, pL=0.0002016344934


  p=0.001, shots=4,400,000, accepted=2,666,989, errors=1025, acceptance=60.613386%, pL=0.0003843285443


...

  p=0.0012, shots=3,000,000, accepted=1,645,845, errors=1002, acceptance=54.861500%, pL=0.000608805811


  p=0.0014, shots=2,400,000, accepted=1,190,964, errors=1003, acceptance=49.623500%, pL=0.0008421749104


  p=0.0017, shots=1,600,000, accepted=684,549, errors=1034, acceptance=42.784312%, pL=0.001510483545


  p=0.002, shots=1,200,000, accepted=442,168, errors=1056, acceptance=36.847333%, pL=0.002388232527


X-error fits:
  unweighted: pL ≈ 4.53e+04 p^2.701
  error-weighted: pL ≈ 5.06e+04 p^2.718


## Fresh Monte Carlo for the authors' full branch

This records an independent simulation of the authors' exact
`full_post_selection=True` circuit. The helper shown below builds that circuit,
asserts that all 135 detectors are postselected, verifies that PyMatching makes
no logical correction for the accepted all-zero syndrome, and samples the
undecomposed Stim detector error model with fixed independent seeds.

The high-statistics run uses the six probabilities from $8\times10^{-4}$
through $2\times10^{-3}$, omitting the two smallest Figure 9 probabilities. It
targets 50 logical errors in each of ten independent shards, producing 500--506
logical errors per point. The quoted uncertainty is the standard deviation of
20,000 Poisson counting-noise bootstrap refits using the same unweighted
nonlinear fit of $C p^\alpha$ that reproduces the paper's reported exponents
from its released counts.

In [4]:
print(inspect.getsource(run_true_job))

def run_job(job: Job) -> Result:
    initial_value = InitialValue.Plus if job.probe == "+" else InitialValue.Zero
    experiment = SteanePlusSurfaceCode(
        QubitMapping(30, 30),
        5,
        initial_value,
        SteaneSyndromeExtractionPattern.ZXZ,
        job.p,
        True,
    )
    experiment.run()
    circuit = experiment.circuit.circuit
    if len(experiment.circuit.detectors_for_post_selection) != circuit.num_detectors:
        raise AssertionError("full_post_selection=True did not select every detector")

    matching = pymatching.Matching.from_detector_error_model(
        circuit.detector_error_model(decompose_errors=True)
    )
    if np.any(matching.decode(np.zeros(circuit.num_detectors, dtype=np.uint8))):
        raise AssertionError("PyMatching applies a logical correction to the accepted zero syndrome")
    sampler = circuit.detector_error_model(decompose_errors=False).compile_sampler(seed=job.seed)
    shots = 0
    accepted = 0
    errors = 0
    while e

In [5]:
true_scaling_results = {
    "+": [
        (0.0008, 3_954_000_000, 1_580_710_343, 504),
        (0.0010, 2_242_000_000, 712_784_921, 503),
        (0.0012, 1_640_000_000, 414_613_001, 503),
        (0.0014, 1_420_000_000, 285_469_396, 506),
        (0.0017, 996_000_000, 142_038_124, 505),
        (0.0020, 884_000_000, 89_404_785, 506),
    ],
    "0": [
        (0.0008, 19_630_000_000, 7_866_015_738, 500),
        (0.0010, 11_244_000_000, 3_585_192_687, 501),
        (0.0012, 8_066_000_000, 2_046_361_433, 500),
        (0.0014, 7_132_000_000, 1_439_832_353, 501),
        (0.0017, 5_422_000_000, 777_027_748, 501),
        (0.0020, 4_812_000_000, 489_572_832, 501),
    ],
}

for probe, error_name in (("+", "Z"), ("0", "X")):
    rows = true_scaling_results[probe]
    print(f"{error_name}-error probe ({probe}), fresh full-postselection run:")
    for p, shots, accepted, errors in rows:
        print(
            f"  p={p:.4g}, shots={shots:,}, accepted={accepted:,}, "
            f"errors={errors}, acceptance={accepted / shots:.6%}, "
            f"pL={errors / accepted:.10g}"
        )
    p_values = np.array([row[0] for row in rows])
    accepted_counts = np.array([row[2] for row in rows])
    error_counts = np.array([row[3] for row in rows])
    rates = error_counts / accepted_counts
    fit, covariance = curve_fit(
        true_power_law,
        p_values,
        rates,
        p0=(1000, 3),
        maxfev=100_000,
    )
    rng = np.random.default_rng(20260809 + (probe == "0"))
    bootstrap_exponents = []
    for _ in range(20_000):
        bootstrap_rates = rng.poisson(error_counts) / accepted_counts
        bootstrap_fit, _ = curve_fit(
            true_power_law,
            p_values,
            bootstrap_rates,
            p0=fit,
            maxfev=10_000,
        )
        bootstrap_exponents.append(bootstrap_fit[1])
    bootstrap_exponents = np.asarray(bootstrap_exponents)
    bootstrap_se = np.std(bootstrap_exponents, ddof=1)
    bootstrap_interval = np.quantile(bootstrap_exponents, [0.025, 0.975])
    print(
        f"  six-point paper-style fit: pL ≈ {fit[0]:.3g} p^{fit[1]:.3f} "
        f"(bootstrap 1σ={bootstrap_se:.3f}, "
        f"95% CI=[{bootstrap_interval[0]:.3f}, {bootstrap_interval[1]:.3f}])"
    )
    print(
        f"  six-point totals: shots={sum(row[1] for row in rows):,}, "
        f"logical errors={sum(row[3] for row in rows)}"
    )

Z-error probe (+), fresh full-postselection run:
  p=0.0008, shots=3,954,000,000, accepted=1,580,710,343, errors=504, acceptance=39.977500%, pL=3.188439946e-07
  p=0.001, shots=2,242,000,000, accepted=712,784,921, errors=503, acceptance=31.792369%, pL=7.056827174e-07
  p=0.0012, shots=1,640,000,000, accepted=414,613,001, errors=503, acceptance=25.281281%, pL=1.213179516e-06
  p=0.0014, shots=1,420,000,000, accepted=285,469,396, errors=506, acceptance=20.103479%, pL=1.772519251e-06
  p=0.0017, shots=996,000,000, accepted=142,038,124, errors=505, acceptance=14.260856%, pL=3.555383483e-06
  p=0.002, shots=884,000,000, accepted=89,404,785, errors=506, acceptance=10.113663%, pL=5.659652333e-06


  six-point paper-style fit: pL ≈ 1.14e+03 p^3.077 (bootstrap 1σ=0.131, 95% CI=[2.832, 3.344])
  six-point totals: shots=11,136,000,000, logical errors=3027
X-error probe (0), fresh full-postselection run:
  p=0.0008, shots=19,630,000,000, accepted=7,866,015,738, errors=500, acceptance=40.071400%, pL=6.35645817e-08
  p=0.001, shots=11,244,000,000, accepted=3,585,192,687, errors=501, acceptance=31.885385%, pL=1.397414431e-07
  p=0.0012, shots=8,066,000,000, accepted=2,046,361,433, errors=500, acceptance=25.370214%, pL=2.443361138e-07
  p=0.0014, shots=7,132,000,000, accepted=1,439,832,353, errors=501, acceptance=20.188339%, pL=3.479571764e-07
  p=0.0017, shots=5,422,000,000, accepted=777,027,748, errors=501, acceptance=14.331017%, pL=6.447646191e-07
  p=0.002, shots=4,812,000,000, accepted=489,572,832, errors=501, acceptance=10.173999%, pL=1.023341099e-06


  six-point paper-style fit: pL ≈ 73 p^2.909 (bootstrap 1σ=0.127, 95% CI=[2.671, 3.167])
  six-point totals: shots=56,306,000,000, logical errors=3004


## How the fault pairs are found

For each elementary error mechanism (i), the undecomposed Stim detector error
model supplies a binary detector mask (d_i) and logical mask \(\ell_i\). A
two-fault combination ((i,j)) has

\[
d_{ij}=d_i\mathbin{\mathrm{XOR}}d_j,\qquad
\ell_{ij}=\ell_i\mathbin{\mathrm{XOR}}\ell_j.
\]

It is accepted when (d_{ij}) has no overlap with the authors' postselection
mask. The search groups accepted pairs by their combined detector mask and
looks for two pairs with the same mask but opposite logical masks. Stim then
maps each DEM effect back to a representative physical Pauli fault and exact
circuit instruction. The witnesses below use distinct circuit locations, so
each combination is a physically possible two-fault event with nonzero
(O(p^2)) probability.

### Elementary-effect extraction from the authors' circuit

In [6]:
print(inspect.getsource(extract_fault_effects))

def extract_fault_effects(circuit: stim.Circuit) -> list[FaultEffect]:
    """Extract every elementary fault's detector and logical effect."""
    detector_error_model = circuit.detector_error_model(
        decompose_errors=False,
        flatten_loops=True,
    )
    model_effects = [
        dem_targets_to_masks(instruction.targets_copy())
        for instruction in detector_error_model
        if instruction.type == "error"
    ]
    explanations = circuit.explain_detector_error_model_errors(
        reduce_to_one_representative_error=True,
    )
    if len(model_effects) != len(explanations):
        raise AssertionError("Detector-error effects and explanations have different lengths")

    effects: list[FaultEffect] = []
    for model_effect, explanation in zip(model_effects, explanations):
        explained_effect = dem_targets_to_masks(
            [term.dem_target for term in explanation.dem_error_terms]
        )
        if model_effect != explained_effect:
            raise 

### Exhaustive same-syndrome search

In [7]:
print(inspect.getsource(rediscover_ambiguity))

def rediscover_ambiguity(
    effects: list[FaultEffect],
    postselection_mask: int,
) -> tuple[tuple[int, int], tuple[int, int]]:
    """Find two accepted two-fault pairs with one syndrome and opposite logicals."""
    first_pair_by_syndrome_and_logical: dict[tuple[int, int], tuple[int, int]] = {}
    for first_index, first in enumerate(effects):
        for second_index in range(first_index + 1, len(effects)):
            second = effects[second_index]
            detector_mask = first.detector_mask ^ second.detector_mask
            if detector_mask & postselection_mask:
                continue
            logical_mask = first.logical_mask ^ second.logical_mask
            opposite = first_pair_by_syndrome_and_logical.get(
                (detector_mask, logical_mask ^ 1)
            )
            if opposite is not None:
                return opposite, (first_index, second_index)
            first_pair_by_syndrome_and_logical.setdefault(
                (detector_mask, logical_

## Independently rediscovered fault pairs

This reruns the search on each authors' `full_post_selection=False` circuit.
`print_ambiguity` verifies acceptance, equal detector syndromes, and opposite
logical outcomes before printing Stim's physical circuit locations.

In [8]:
for probe in ("+", "0"):
    experiment = build_experiment(probe, full_post_selection=False)
    effects = extract_fault_effects(experiment.circuit)
    pairs = rediscover_ambiguity(effects, experiment.postselection_mask)
    verify_ambiguity(effects, experiment.postselection_mask, pairs)
    print(f"\n{probe} probe: rediscovered {pairs}")
    print_ambiguity(probe, effects, experiment.postselection_mask, pairs)


+ probe: rediscovered ((186, 246), (206, 341))

+ probe: accepted same-syndrome/opposite-logical witness
  common detection events: [16, 22, 25, 56]
  logical difference: L0=1
  pair 1: DEM effect indices (186, 246), L0=0
    combined detection events: [16, 22, 25, 56]
    fault 1: effect index 186, D=[16, 17], L0=0
      ExplainedError {
          dem_error_terms: D16 D17
          CircuitErrorLocation {
              flipped_pauli_product: Y39[coords 4,18]*Y263[coords 5,17]
              Circuit location stack trace:
                  (after 8 TICKs)
                  at instruction #901 (DEPOLARIZE2) in the circuit
                  at targets #1 to #2 of the instruction
                  resolving to DEPOLARIZE2(0.001) 39[coords 4,18] 263[coords 5,17]
          }
      }
    fault 2: effect index 246, D=[17, 22, 25, 56], L0=0
      ExplainedError {
          dem_error_terms: D17 D22 D25 D56
          CircuitErrorLocation {
              flipped_pauli_product: X250[coords 3,21]*Y40


0 probe: rediscovered ((185, 1350), (212, 818))

0 probe: accepted same-syndrome/opposite-logical witness
  common detection events: [16, 77]
  logical difference: L0=1
  pair 1: DEM effect indices (185, 1350), L0=1
    combined detection events: [16, 77]
    fault 1: effect index 185, D=[16], L0=0
      ExplainedError {
          dem_error_terms: D16
          CircuitErrorLocation {
              flipped_pauli_product: X24[coords 2,18]
              Circuit location stack trace:
                  (after 0 TICKs)
                  at instruction #502 (X_ERROR) in the circuit
                  at target #1 of the instruction
                  resolving to X_ERROR(0.001) 24[coords 2,18]
          }
      }
    fault 2: effect index 1350, D=[77], L0=1
      ExplainedError {
          dem_error_terms: D77 L0
          CircuitErrorLocation {
              flipped_pauli_product: X233[coords 1,17]
              Circuit location stack trace:
                  (after 18 TICKs)
                

## Interpretation

Across the finite Figure 9 window, the non-full circuit has an effective fitted
exponent of approximately (2.7) for both probes. Separately, for each probe,
one accepted detector syndrome is compatible with both logical classes at
two-fault order. A decoder that receives only this syndrome must choose one
class and therefore fails on the other. This establishes a nonzero asymptotic
(O(p^2)) contribution for the authors' `full_post_selection=False` circuit.
Full postselection rejects these detector events and addresses a different
mode. The fresh high-statistics full-branch run independently reproduces the
approximately 31.8% retention at $p=10^{-3}$ and gives six-point paper-style
fits of $\alpha_Z=3.077\pm0.131$ and $\alpha_X=2.909\pm0.126$, where the quoted
one-standard-deviation uncertainties come from counting-noise bootstrap
refits.